# 5 - Fit Statistical Models

**Purpose**: Fit Cox proportional hazards (OS and PFS) and logistic regression
models for the CIDER Score across all lymphocyte groups, broad cohorts,
and CDR3 metric types. Two covariate specifications: unadjusted ("None") and
fully adjusted ("TMB/Clinical/Purity" — TMB, purity, age, sex, prior
systemic therapy, prior radiotherapy, and biopsy structure). Also fits
mechanism subgroup models (grouped by ICI treatment mechanism) used in the
mechanism forest plot supplement.

The cohort-stratified fits also split the residual "Other" cohort into **Non-CRC MSI** and **Other MSS** (by `msi_high`) before fitting -- two separately-fit strata instead of one combined "Other" -- see `helpers/shortcuts.r::split_other_by_msi()`. Same split used for display purposes in Table 1 and Figures 3/4.

**Standardization**: the CDR3/CIDER score predictor (`cider_log2`) is
`log2(x + 1)` only -- NOT standardized -- so its effect size is "per
doubling of CIDER score + 1" everywhere a forest plot reports it, matching
the convention used in `1_figures/15_ici_specificity.ipynb` and
`1_figures/20_ici_specificity_main.ipynb` (Figure 6). TMB, age, and purity
remain standardized (`scale()`) once across all ICI-treated patients
pooled -- the same scaling is shared by the cohort models, the pan-ICI
("ICI-Treated") row, and the mechanism-subgroup models below, so those
covariates are on identical per-SD units everywhere in this notebook. They
are adjustment covariates only (not effects of interest in their own
right), so a single pooled scaling is sufficient; only the CDR3 score's
adjusted-vs-unadjusted comparison is interpreted directly.

| | |
|---|---|
| **Inputs** | `CIDER_PROJ_DIR/cpi_ready.rds` |
| **Outputs** | `CIDER_PROJ_DIR/models_ready.rds` (cohort + mechanism rows, unified schema) |
| **Run before** | `1_figures/8_forest.ipynb` |


In [1]:
source(paste0(dirname(getwd()), "/helpers/map.r"))
source(paste0(dirname(getwd()), "/helpers/shortcuts.r"))
source(paste0(dirname(getwd()), "/helpers/config.r"))

assert_files_exist(
  "CPI ready data" = paste0(CIDER_PROJ_DIR, "cpi_ready.rds")
)

# 0 - Read and prepare data

In [ ]:
# tidyverse is already loaded by shortcuts.r
library(broom)
library(survival)

# CIDER scores and TMB are both on ratio scales in cpi_ready.rds (CIDER:
# cider_score_depth_adj/cider_score_adj; TMB: raw tmbPerMb, not pre-logged --
# see 0_prepare/3_meta.ipynb). log2(x + 1) handles right-skew/zero-inflation
# for both, and a one-unit change in the logged value corresponds to a
# doubling of the raw value for either covariate (same convention used
# throughout 1_figures/, e.g. 9_roc.ipynb's combined TMB + CIDER model).
# CIDER (cider_log2) is used directly, unstandardized -- matching
# 15_ici_specificity.ipynb/20_ici_specificity_main.ipynb's convention so
# every forest plot in the paper reports the CIDER effect on the same
# units. TMB additionally goes through scale() since it's an adjustment
# covariate only here, not an effect of interest in its own right.
#
# metric_type labels:
#   "CIDER Score"                  — depth adjusted only (primary metric)
#   "CIDER Score (Purity Adjusted)" — depth + purity corrected (sensitivity analysis)
#
# Comparing these two directly shows the marginal contribution of purity correction.
#
# Standardization: every scaled_* column below (TMB, age, purity -- NOT
# cider_log2, see above) uses ONE common scaling computed across all
# ICI-treated patients pooled — NOT re-scaled per cohort_broad.
# cpi_ready.rds duplicates each ICI patient once under their tumour-type
# cohort and once under the "ICI-Treated" pan-cancer row (see 4_base.ipynb);
# that duplication is uniform (every patient appears exactly twice) so it
# doesn't bias the pooled mean/SD used here. A single shared scaling keeps
# every model in this notebook — cohort-specific, pan-ICI, and
# mechanism-subgroup (below) — on the same per-SD units, which matters
# because these covariates are adjustment terms only (we care whether the
# CIDER Score effect holds after adjustment, not the covariates' own effect
# sizes, so cohort-specific SD comparability for TMB/age/purity isn't
# needed).

ready <-
  readRDS(paste0(CIDER_PROJ_DIR, "cpi_ready.rds")) |>
  select(sampleId, cohort_broad, cohort_broad_go, msi_high, gp, response, dcb,
         osDays, osEvent, pfsEventDays, pfsEvent,
         age, sex, pretreated, priorRadiotherapy, biopsyStructure,
         tmbPerMb, purity, cider_score_depth_adj, cider_score_adj,
         treatment, mechanism)

# Split cohort_broad's residual "Other" stratum into Non-CRC MSI / Other
# (MSS) BEFORE any model is fit below -- the per-cohort forest plot
# (1_figures/8_forest.ipynb, fed by this notebook's models_ready.rds)
# otherwise shows one "Other" row mixing MSI-high and microsatellite-stable
# patients from many different tumor types. Unlike the same split in
# 1_response.ipynb/2_survival.ipynb (display-only relabeling of an already-
# built plot), this is a real refit: the former single "Other" regression
# becomes two independently-fit strata, each on a smaller N -- see
# helpers/shortcuts.r::split_other_by_msi(). SKCM/NSCLC/BLCA/CRC-MSI/
# ICI-Treated are unaffected and keep their existing fit/N.
# Level order: BROAD_LEVELS_OTHER_SPLIT (helpers/config.r), same shared
# order used everywhere else this split appears -- doesn't change this
# plot's own visual order (8_forest.ipynb re-sorts strata by N), kept
# consistent anyway so there's one source of truth for the split's order.
cohort_other_split_levels <- c(
  BROAD_LEVELS_OTHER_SPLIT,
  intersect("Other (Unknown MSI)", unique(split_other_by_msi(ready$cohort_broad, ready$msi_high)))
)
ready <- ready |>
  mutate(cohort_broad = factor(
    split_other_by_msi(cohort_broad, msi_high),
    levels = cohort_other_split_levels
  )) |>
  group_by(cohort_broad) |>
  mutate(n_broad = n_distinct(sampleId)) |>
  ungroup() |>
  mutate(
    cohort_broad_go = {
      lkp <- distinct(pick(cohort_broad, n_broad)) |> arrange(cohort_broad)
      factor(
        paste0(cohort_broad, " (N = ", n_broad, ")"),
        levels = paste0(lkp$cohort_broad, " (N = ", lkp$n_broad, ")")
      )
    }
  ) |>
  select(-n_broad)

ready <- ready |>
  pivot_longer(
    cols      = c(cider_score_depth_adj, cider_score_adj),
    names_to  = "metric_type",
    values_to = "val"
  ) |>
  mutate(metric_type = factor(
    case_when(
      metric_type == "cider_score_depth_adj" ~ "CIDER Score",
      metric_type == "cider_score_adj"       ~ "CIDER Score (Purity Adjusted)",
      TRUE ~ metric_type
    ),
    levels = c("CIDER Score", "CIDER Score (Purity Adjusted)")
  )) |>
  mutate(
    cider_log2  = log2(val + 1),
    scaled_tmb  = as.vector(scale(log2(tmbPerMb + 1))),
    # Clinical covariates
    # pretreated is numeric/character 1/0 in cpi_ready.rds (confirmed against
    # the data), not text — tolower(pretreated) %in% c("true","yes") never
    # matched, so pretreated_n was silently constant 0 for every patient in
    # every fitted model below. == coerces correctly whether pretreated is
    # stored as numeric or character "1"/"0".
    #
    # priorRadiotherapy is loaded identically to pretreated in 3_meta.ipynb
    # (same "NULL" -> NA passthrough from clinical_go.txt) and is likely the
    # same 1/0 encoding, but that hasn't been directly confirmed against the
    # data the way pretreated has — left on the old tolower()-based check
    # pending confirmation, since silently changing it without checking would
    # just be a different guess rather than a verified fix.
    scaled_age           = as.vector(scale(as.numeric(age))),
    sex_female           = as.integer(tolower(sex) %in% c("female", "f")),
    pretreated_n         = as.integer(pretreated == 1),
    prior_radiotherapy_n = as.integer(tolower(priorRadiotherapy) %in% c("true", "yes")),
    scaled_purity        = as.vector(scale(purity)),
    # biopsyStructure arrives from 3_meta.ipynb already collapsed to
    # BIOPSY_STRUCTURE_LEVELS (helpers/config.r) — carried through as-is;
    # coxph()/glm() dummy-code multi-level factor covariates automatically.
    biopsy_structure      = biopsyStructure
  )


# 1 - Define model specifications

In [ ]:
# Two covariate specifications:
#   "None"                — CIDER Score alone (primary estimate)
#   "TMB/Clinical/Purity" — + TMB, purity, age, sex, prior therapy, prior
#                            radiotherapy, and biopsy structure (fully adjusted)

cox_specs <- list(
  list(type = "cox", covariates = "None",
       formula = Surv(osDays, osEvent) ~ cider_log2),
  list(type = "cox", covariates = "TMB/Clinical/Purity",
       formula = Surv(osDays, osEvent) ~ cider_log2 + scaled_tmb + scaled_purity +
                 scaled_age + sex_female + pretreated_n + prior_radiotherapy_n + biopsy_structure)
)

pfs_cox_specs <- list(
  list(type = "cox", covariates = "None",
       formula = Surv(pfsEventDays, pfsEvent) ~ cider_log2),
  list(type = "cox", covariates = "TMB/Clinical/Purity",
       formula = Surv(pfsEventDays, pfsEvent) ~ cider_log2 + scaled_tmb + scaled_purity +
                 scaled_age + sex_female + pretreated_n + prior_radiotherapy_n + biopsy_structure)
)

logistic_specs <- list(
  list(type = "logistic", covariates = "None",
       formula = response ~ cider_log2),
  list(type = "logistic", covariates = "TMB/Clinical/Purity",
       formula = response ~ cider_log2 + scaled_tmb + scaled_purity +
                 scaled_age + sex_female + pretreated_n + prior_radiotherapy_n + biopsy_structure)
)

dcb_logistic_specs <- list(
  list(type = "logistic", covariates = "None",
       formula = dcb ~ cider_log2),
  list(type = "logistic", covariates = "TMB/Clinical/Purity",
       formula = dcb ~ cider_log2 + scaled_tmb + scaled_purity +
                 scaled_age + sex_female + pretreated_n + prior_radiotherapy_n + biopsy_structure)
)

all_specs <- c(cox_specs, pfs_cox_specs, logistic_specs, dcb_logistic_specs)


# 2 - Model runner

In [4]:
# Fit one model spec across all (gp, strata, metric_type) groups.
# strata = cohort_broad_go label (e.g. "NSCLC (N = 120)", "ICI-Treated (N = 656)").
# Returns a tidy tibble with metadata columns appended, including n_obs
# (number of observations actually used in each regression).
model_runner <- function(spec) {
  fit_one <- function(d) {
    if (spec$type == "cox") {
      coxph(spec$formula, data = d)
    } else {
      glm(spec$formula, data = d, family = "binomial")
    }
  }

  get_nobs <- function(fit) {
    if (inherits(fit, "coxph")) fit$n else nobs(fit)
  }

  ready |>
    group_by(gp, cohort_broad_go, metric_type) |>
    nest() |>
    mutate(
      fit     = map(data, fit_one),
      n_obs   = map_int(fit, get_nobs),
      results = map(fit, tidy, conf.int = TRUE)
    ) |>
    unnest(results) |>
    transmute(
      metric_type,
      strata      = cohort_broad_go,
      strata_type = "cohort",
      gp,
      term, estimate, std.error, statistic, p.value, conf.low, conf.high,
      n_obs,
      model      = if (spec$type == "cox") "Cox-PH" else "Logistic Regression",
      endpoint   = {
        f_str <- paste(deparse(spec$formula), collapse = " ")
        if (grepl("pfsEventDays", f_str)) "PFS"
        else if (spec$type == "cox") "OS"
        else if (grepl("^dcb ", f_str)) "DCB"
        else "BOR"
      },
      predictors = attr(terms(spec$formula), "term.labels")[1],
      covariates = spec$covariates,
      formula    = paste(deparse(spec$formula), collapse = " ")
    ) |>
    ungroup()
}


# 3 - Fit all models and curate results

In [ ]:
# Reversed GP_LEVELS for bottom-to-top forest plot ordering
gp_levels <- rev(GP_LEVELS)

results <-
  bind_rows(lapply(all_specs, model_runner)) |>
  dplyr::filter(term == "cider_log2") |>
  mutate(
    covariates = factor(covariates, levels = c("None", "TMB/Clinical/Purity")),
    gp         = factor(gp, levels = gp_levels)
  )

# 4 - Save

In [6]:
# Save cohort results now as a fail-safe.
# Cell 13 will append mechanism results and overwrite this file.
saveRDS(results, paste0(CIDER_PROJ_DIR, "models_ready.rds"))


# 5 - Mechanism subgroup models

Fits the same model specifications as above but groups by
`gp × mechanism_group × metric_type` instead of cohort.
Results use `strata_type = "mechanism"` and are appended to
`models_ready.rds` alongside the cohort rows.


In [ ]:
# ── Derive mechanism group ─────────────────────────────────────────────────────
# Uses the `mechanism` column carried through from cpi_ready.rds (assigned in
# 3_meta.ipynb from clinical_go.txt's own `mechanism` field, with one
# hand-curated override for the Ipilimumab/Nivolumab combo) -- the SAME
# field Table S5 (1_figures/11e_treatment_by_mechanism_response.ipynb) and
# Table 1 use. This used to be re-derived independently here from the raw
# `treatment` string via ICI_TREATMENT_NORM + ICI_MECHANISM_MAP
# (helpers/config.r), which could silently disagree with Table S5/Table 1
# for any patient whose treatment string wasn't recognized by that lookup
# (they'd fall back to "Multiple mechanism" here even when the curated
# clinical `mechanism` field said otherwise) -- exactly the kind of
# cross-table N mismatch that prompted this change. Using the same source
# column everywhere guarantees the forest plot's per-mechanism Ns match
# Table S5's exactly.
#
# IMPORTANT: filter cohort_broad != "ICI-Treated" before any mechanism analysis.
# cpi_ready.rds contains each ICI patient TWICE — once under their specific
# tumour-type cohort and once under "ICI-Treated" (the pan-cancer duplicate
# created in 4_base.ipynb for cohort-stratified forest plots). Without the
# filter every patient would appear twice, halving standard errors and
# inflating test statistics.
#
# cider_log2 / scaled_tmb / scaled_age / scaled_purity are inherited as-is
# from `ready` (cider_log2 is just log2(x+1), unstandardized; scaled_tmb/
# scaled_age/scaled_purity use one common scaling across all ICI-treated
# patients, computed above) — NOT re-scaled here. This keeps
# mechanism-subgroup models on the exact same units as the cohort models
# above, so the two forest plots stay directly comparable.

# Mechanism levels are derived from whatever distinct `mechanism` strings are
# actually present in cpi_ready.rds -- via fct_pin_last(factor(mechanism),
# "Multiple mechanism") (helpers/shortcuts.r), the EXACT SAME call Table S5
# and Table 1 use (11e_treatment_by_mechanism_response.ipynb,
# 11_demographic_table.ipynb) -- rather than a hand-typed whitelist of
# literal strings.
#
# A hand-typed whitelist (previously c("Anti-PD1", "Multiple mechanism",
# "Anti-CTLA4/Anti-PD1", "Anti-PDL1", "Anti-CTLA4")) is fragile: it assumes
# the exact spelling of every raw clinical_go.txt category, and
# factor(mechanism, levels = <whitelist>) silently turns anything that
# doesn't match byte-for-byte into NA -- with no warning, that patient still
# gets fit (group_by() keeps NA as its own group) and ends up as an
# unlabeled NA row in the mechanism forest plot instead of being dropped
# and reported. This is exactly what caused "Anti-CTLA4" and "Anti-PDL1" to
# vanish from the plot: the real clinical_go.txt strings don't necessarily
# match those guessed spellings byte-for-byte. Deriving levels straight from
# the data removes that whole class of mismatch and guarantees this
# notebook's categories always match Table S5/Table 1's, whatever the
# underlying strings actually are.
mech_ready <- ready |>
  dplyr::filter(cohort_broad != "ICI-Treated", !is.na(mechanism)) |>
  dplyr::mutate(mechanism_group = fct_pin_last(factor(mechanism), "Multiple mechanism"))

MECHANISM_LEVELS <- levels(mech_ready$mechanism_group)
message("Mechanism levels (from data): ", paste(MECHANISM_LEVELS, collapse = ", "))

n_mech_excluded <- ready |>
  dplyr::filter(cohort_broad != "ICI-Treated", is.na(mechanism)) |>
  dplyr::distinct(sampleId) |>
  nrow()
if (n_mech_excluded > 0) {
  message(n_mech_excluded, " ICI-treated patient(s) excluded from mechanism models: missing `mechanism`.")
}

message("mech_ready: ", nrow(mech_ready), " rows, ",
        dplyr::n_distinct(mech_ready$sampleId), " unique patients")

# ── Mechanism model specs ──────────────────────────────────────────────────────
mech_specs <- list(
  list(type = "cox",      covariates = "None",
       formula = Surv(osDays, osEvent) ~ cider_log2),
  list(type = "cox",      covariates = "TMB/Clinical/Purity",
       formula = Surv(osDays, osEvent) ~ cider_log2 + scaled_tmb + scaled_purity +
                 scaled_age + sex_female + pretreated_n + prior_radiotherapy_n + biopsy_structure),
  list(type = "cox",      covariates = "None",
       formula = Surv(pfsEventDays, pfsEvent) ~ cider_log2),
  list(type = "cox",      covariates = "TMB/Clinical/Purity",
       formula = Surv(pfsEventDays, pfsEvent) ~ cider_log2 + scaled_tmb + scaled_purity +
                 scaled_age + sex_female + pretreated_n + prior_radiotherapy_n + biopsy_structure),
  list(type = "logistic", covariates = "None",
       formula = response ~ cider_log2),
  list(type = "logistic", covariates = "TMB/Clinical/Purity",
       formula = response ~ cider_log2 + scaled_tmb + scaled_purity +
                 scaled_age + sex_female + pretreated_n + prior_radiotherapy_n + biopsy_structure),
  list(type = "logistic", covariates = "None",
       formula = dcb ~ cider_log2),
  list(type = "logistic", covariates = "TMB/Clinical/Purity",
       formula = dcb ~ cider_log2 + scaled_tmb + scaled_purity +
                 scaled_age + sex_female + pretreated_n + prior_radiotherapy_n + biopsy_structure)
)

# ── Mechanism model runner ─────────────────────────────────────────────────────
# Groups by gp x mechanism_group x metric_type.
# strata_type = "mechanism"; strata uses mechanism_group's own values
# directly (the same category names as Table S5/Table 1 -- see the
# "Derive mechanism group" comment above).
# Drops groups with < 10 non-NA observations; returns NULL safely on error.
mech_model_runner <- function(spec) {
  outcome_var <- {
    f_str <- paste(deparse(spec$formula), collapse = " ")
    if (grepl("^dcb ", f_str)) "dcb"
    else if (spec$type == "logistic") "response"
    else NULL
  }

  fit_one <- function(d) {
    if (!is.null(outcome_var)) d <- d[!is.na(d[[outcome_var]]), ]
    if (nrow(d) < 10) return(NULL)
    tryCatch(
      if (spec$type == "cox")
        coxph(spec$formula, data = d)
      else
        glm(spec$formula, data = d, family = "binomial"),
      error = function(e) NULL
    )
  }

  tidy_safe <- function(fit) {
    if (is.null(fit)) return(NULL)
    tryCatch(broom::tidy(fit, conf.int = TRUE), error = function(e) NULL)
  }

  get_nobs <- function(fit) {
    if (is.null(fit)) return(NA_integer_)
    if (inherits(fit, "coxph")) fit$n else nobs(fit)
  }

  nested <- mech_ready |>
    dplyr::group_by(gp, mechanism_group, metric_type) |>
    tidyr::nest() |>
    dplyr::mutate(
      fit     = purrr::map(data, fit_one),
      n_obs   = purrr::map_int(fit, get_nobs),
      results = purrr::map(fit, tidy_safe)
    ) |>
    dplyr::filter(!purrr::map_lgl(results, is.null)) |>
    dplyr::filter(purrr::map_int(results, nrow) > 0)

  if (nrow(nested) == 0) {
    message("No fits for: ", spec$covariates)
    return(NULL)
  }

  nested |>
    tidyr::unnest(results) |>
    dplyr::filter(term == "cider_log2") |>
    dplyr::transmute(
      metric_type,
      strata      = factor(as.character(mechanism_group), levels = MECHANISM_LEVELS),
      strata_type = "mechanism",
      gp,
      term, estimate, std.error, statistic, p.value, conf.low, conf.high,
      n_obs,
      model      = if (spec$type == "cox") "Cox-PH" else "Logistic Regression",
      endpoint   = {
        f_str <- paste(deparse(spec$formula), collapse = " ")
        if (grepl("pfsEventDays", f_str)) "PFS"
        else if (spec$type == "cox") "OS"
        else if (grepl("^dcb ", f_str)) "DCB"
        else "BOR"
      },
      covariates = spec$covariates
    )
}

# ── Fit mechanism models ───────────────────────────────────────────────────────
mech_results_list <- Filter(Negate(is.null), lapply(mech_specs, mech_model_runner))

if (length(mech_results_list) == 0) {
  warning("No mechanism models fitted. Check that 'treatment' column exists in cpi_ready.rds.")
} else {
  mech_results <- dplyr::bind_rows(mech_results_list) |>
    dplyr::mutate(
      gp         = factor(gp, levels = GP_LEVELS),
      covariates = factor(covariates, levels = c("None", "TMB/Clinical/Purity")),
      ci_crosses_zero = conf.low < 0 & conf.high > 0
    )

  message("Mechanism models fitted: ", nrow(mech_results), " rows")

  # ── Combine cohort + mechanism results and save as unified models_ready.rds ──
  all_results <- dplyr::bind_rows(results, mech_results)
  saveRDS(all_results, paste0(CIDER_PROJ_DIR, "models_ready.rds"))
  message("Saved unified models_ready.rds (",
          nrow(all_results), " rows: ",
          sum(all_results$strata_type == "cohort"), " cohort, ",
          sum(all_results$strata_type == "mechanism"), " mechanism)")
}
